# CodeGen — NL → MQL fine-tuning template (Unsloth, Kaggle or Colab T4)

A plain, reusable QLoRA fine-tuning notebook for the capstone. It:

1. loads **Qwen2.5-Coder-3B-Instruct** in 4-bit,
2. runs a few held-out questions through the **base model** ("before"),
3. fine-tunes a LoRA adapter on the SFT data built from the approved corpus,
4. runs the **same questions** again ("after"),
5. saves the adapter.

**What this notebook is not.** It is not an experiment and it produces no benchmark number. Reported NL→MQL
accuracy must come from execution on MongoDB Atlas with the project's scoring scripts. The checks here
(dev loss, "is the output valid JSON with a known collection") only tell you the run worked.

### Kaggle setup
- **Accelerator: GPU T4 x2.** Not P100: Unsloth needs a GPU with compute capability 7.0+, and the P100 is 6.0.
  Only one of the two T4s is used.
- **Internet: On** (Settings panel, needs a phone-verified account).
- **Data:** create a private Kaggle Dataset from the `approved_v2_1` folder and add it with "Add Input".
  The notebook finds the files under `/kaggle/input` itself, so the dataset name doesn't matter.
- For the full run, use **Save Version → Save & Run All (Commit)**. It runs in the background (up to 12 h)
  and keeps `/kaggle/working` as the version's output. An interactive session stops if you leave it idle.

### Colab setup
`Runtime → Change runtime type → T4 GPU`; put the two files in Drive (path in Config) or upload when asked.

### About the install
The install cell **does not replace Kaggle's or Colab's torch**. Upgrading or pinning torch,
transformers and similar packages over the preinstalled ones is what causes most dependency breakage.
pip/uv may still print "dependency conflicts" about packages this notebook never uses (tensorflow,
bigframes, google-cloud-*). Those messages are harmless; the check cell after the install shows whether
everything this notebook needs actually imports.

Change things only in the **Config** cell unless you mean to change the recipe.

## 0. Environment (run first, before anything imports torch)

In [1]:
import os
import sys

# setting these env variables BEFORE anything imports torch, otherwise they dont do anything
# kaggle gives 2 T4s but unsloth only trains on 1, so just use gpu 0
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# kaggle has wandb preinstalled and the trainer keeps asking for an api key, so turning it off
os.environ["WANDB_DISABLED"] = "true"
# transformers tries to import kaggle's tensorflow for no reason, this stops it
os.environ["USE_TF"] = "0"
# just stops the tokenizers fork warning spam
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# figure out if we're on kaggle or colab (kaggle always has /kaggle/working)
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = False
if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
    ON_COLAB = True

if ON_KAGGLE:
    platform = "kaggle"
elif ON_COLAB:
    platform = "colab"
else:
    platform = "other"
python_version = sys.version.split()[0]
print("platform:", platform, "| python", python_version)

platform: kaggle | python 3.13.16


## 1. Config

In [2]:
# ======================= CONFIG =======================
# only change stuff in here. everything below just reads these values

# ---- data ----
# the 2 files from the approved_v2_1 folder
# train = what the model learns from, dev = held out (different databases) to check it
TRAIN_FILE = "sft_train.prompt_completion.jsonl"
DEV_FILE = "sft_dev.prompt_completion.jsonl"
# leave it None and it finds the files by itself
# (kaggle: searches /kaggle/input, colab: uses COLAB_DRIVE_DIR)
DATA_DIR = None
COLAB_USE_DRIVE = True   # colab only. set False if you want to upload the 2 files manually instead
COLAB_DRIVE_DIR = "/content/drive/MyDrive/codegen_finetune/approved_v2_1"

# ---- install ----
# right now this just installs whatever the latest unsloth is
# once a run works, check pip_freeze.txt in the output and pin the exact versions here
# so everyone in the team gets the same setup, like:
# UNSLOTH_SPEC = ["unsloth==<version>", "unsloth_zoo==<version>"]
UNSLOTH_SPEC = ["unsloth"]

# ---- model ----
# 3B coder model that unsloth already squished down to 4-bit
# 4-bit = every weight stored in 4 bits instead of 16. kinda like saving a photo as jpeg instead of raw,
# tiny bit of quality loss but ~4x smaller, which is the only reason it fits on a free T4
# want to try 7B? use "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit" (smarter, but slower and barely fits on a T4)
MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
# max tokens in ONE example (system prompt + schema + question + answer all together)
# think of it as the page limit on an answer sheet, anything longer gets dropped
# bigger = more gpu memory. our longest example is ~1.4k tokens (section 7 prints it) so 2048 is safe
MAX_SEQ_LENGTH = 2048

# ---- LoRA ----
# LoRA in one line: we DONT touch the original 3B weights at all (they're frozen),
# we just add small extra matrices next to some layers and only train those.
# analogy: the base model is a printed textbook you cant edit, LoRA is the sticky notes you put on top.

# r (rank) = how big each sticky note is
# bigger r = more room to learn new stuff, but more params, slower, and easier to overfit (just memorise)
# r=16 gives ~30M trainable params, less than 1% of the model. 8 and 32 are also normal values to try
LORA_R = 16

# alpha = how "loud" the sticky notes are compared to the original book
# whatever LoRA learns gets multiplied by alpha / r before it's added, so here it's 32 / 16 = 2x
# too high = the notes drown out what the model already knows, too low = they barely do anything
# project plan says keep alpha = 2 * r, so if you change r change this too (r=8 -> 16, r=32 -> 64)
LORA_ALPHA = 32

# dropout = randomly switch off some % of the LoRA during training so it cant just memorise
# like practicing with some of your notes randomly covered, so you actually learn the thing
# 0 here bcoz unsloth is fastest at 0, and with 1 epoch overfitting isnt a big problem anyway
LORA_DROPOUT = 0.0

# which layers get sticky notes
# q/k/v/o_proj = attention (the part that decides which words to "look at")
# gate/up/down_proj = MLP (the part that does the actual "thinking" on what it looked at)
# all 7 is the usual full set. removing some = fewer params but usually worse results
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---- training ----
# epoch = one full pass over all the training examples. like reading the whole textbook once
# 1 is enough for now. 2 can help but it also starts memorising
NUM_EPOCHS = 1
# set this to like 30 for a quick test run to check nothing crashes. -1 = ignore it and use NUM_EPOCHS
MAX_STEPS = -1

# batch size = how many examples the gpu processes together in one go
# grad accum = how many of those batches we add up before actually updating the weights
# so effective batch = 2 * 4 = 8
# analogy: teacher wants to check 8 papers before giving feedback, but her desk only fits 2,
# so she reads 2, then 2 more... 4 times, and THEN gives feedback. same result, less desk space (gpu memory)
BATCH_SIZE = 2
GRAD_ACCUM = 4

# learning rate = how big a step the model takes each time it updates
# like walking downhill blindfolded: big steps = fast but you overshoot the bottom,
# tiny steps = safe but takes forever. 2e-4 (= 0.0002) is the usual LoRA value
LEARNING_RATE = 2e-4
# warmup = for the first 3% of steps the learning rate slowly ramps up from 0 to 2e-4
# basically stretching before a run so the model doesnt get shocked by big updates at the start
WARMUP_RATIO = 0.03
# cosine = after warmup the learning rate slowly drops to ~0 by the end (follows a cosine curve)
# like slowing your bike down as you get close to where you want to stop
LR_SCHEDULER = "cosine"
# fixed random seed so running it again gives (mostly) the same result
# 3407 is unsloth's default (from that "seed 3407 is all you need" paper lol)
SEED = 3407

# ---- before/after demo ----
# 3 questions from the DEV split (databases the model never sees in training)
# we run these before and after training to compare. can swap in any id from the dev file
DEMO_IDS = ["spider_train_00216", "spider_train_00226", "spider_train_00223"]
# max tokens the model is allowed to write per answer. answers are way shorter, this is just a safety cap
MAX_NEW_TOKENS = 512
# how many dev examples to use for the before/after loss (0 = skip it, saves a few minutes)
DEV_LOSS_SAMPLES = 100

# ---- output ----
RUN_NAME = "qwen25coder3b_lora_r16"   # rename this when you change settings so runs dont overwrite each other
if ON_KAGGLE:
    OUTPUT_DIR = "/kaggle/working/" + RUN_NAME
else:
    OUTPUT_DIR = "/content/outputs/" + RUN_NAME
COLAB_COPY_TO_DRIVE = True   # colab only: also copy the adapter to drive (<drive dir>/../runs/RUN_NAME) so it doesnt vanish

## 2. Install Unsloth (without touching torch)

Tries the safest option first and only loosens it if that fails:
1. keep the installed torch, torchvision, torchaudio **and** numpy;
2. keep only the torch family;
3. let uv choose (torchvision and torchaudio are installed together with torch so they stay matched).

It prints which one worked. If Unsloth is already installed (for example after a kernel restart), it skips.
If a later cell fails with an import error, restart the kernel and run from the top: the install is skipped
the second time and the fresh kernel picks up the new packages.

In [3]:
import importlib.util
import importlib.metadata
import subprocess
import tempfile

# gets the installed version of a package, or None if it's not installed
# the split("+") removes stuff like "+cu130" from the torch version
def get_version(pkg):
    try:
        v = importlib.metadata.version(pkg)
        return v.split("+")[0]
    except importlib.metadata.PackageNotFoundError:
        return None

# runs "uv pip install ..." and returns (did it work, end of the log)
# using uv instead of normal pip bcoz it's waaay faster
def uv_install(args):
    cmd = ["uv", "pip", "install", "--python", sys.executable, "-q"] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    log = result.stderr or result.stdout
    log = log[-3000:]   # only keep the end, the full log is huge
    return result.returncode == 0, log

if importlib.util.find_spec("unsloth") is not None:
    # already installed (happens after a kernel restart), no need to do it again
    print("unsloth already installed:", get_version("unsloth"), "- skipping install")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    # note down the torch versions kaggle/colab already gave us so we dont break them
    torch_versions = {}
    for p in ["torch", "torchvision", "torchaudio"]:
        torch_versions[p] = get_version(p)
    torch_and_numpy = dict(torch_versions)
    torch_and_numpy["numpy"] = get_version("numpy")

    # try 3 times, each time a bit less strict:
    # 1) dont touch torch stuff OR numpy
    # 2) dont touch torch stuff, numpy is allowed to change
    # 3) just let uv do whatever it wants
    attempts = [
        ("keep torch family + numpy", torch_and_numpy),
        ("keep torch family", torch_versions),
        ("unconstrained", None),
    ]

    worked = False
    log = ""
    for label, keep in attempts:
        args = list(UNSLOTH_SPEC)
        if keep is None:
            # if torch gets upgraded, install torchvision + torchaudio along with it so the versions match
            if torch_versions["torch"]:
                args = args + ["torchvision", "torchaudio"]
        else:
            # write a constraints file ("torch==2.x.x" etc) so uv is NOT allowed to change those
            pins = []
            for p, v in keep.items():
                if v:
                    pins.append(p + "==" + v)
            constraints_file = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
            constraints_file.write("\n".join(pins))
            constraints_file.close()
            args = args + ["-c", constraints_file.name]

        ok, log = uv_install(args)
        if ok:
            print("[ok]", label)
            worked = True
            break
        else:
            print("[failed]", label)
            if log.strip():
                print("   ", log.strip().splitlines()[-1])

    if not worked:
        raise RuntimeError("Unsloth install failed with every option. Check that Internet is On (Kaggle settings). "
                           "Last error:\n" + log)

    # double check torch didnt get changed anyway
    changed = {}
    for p, v in torch_versions.items():
        new_v = get_version(p)
        if v and new_v != v:
            changed[p] = (v, new_v)
    if changed:
        print("torch family changed:", changed)
    else:
        print("torch family unchanged")

unsloth already installed: 2026.11.6 - skipping install


### Check the environment

In [4]:
import importlib
import importlib.metadata

# make sure everything actually imports (pip saying "installed" doesnt mean it works lol)
packages = ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"]
problems = []
for name in packages:
    try:
        importlib.import_module(name)
        # the :14s just pads the name so the versions line up
        print(f"{name:14s} {importlib.metadata.version(name)}")
    except Exception as e:
        problems.append(f"{name}: {type(e).__name__}: {e}")

if len(problems) > 0:
    raise RuntimeError("Some packages don't import. Restart the kernel and run from the top (install will be skipped).\n"
                       + "\n".join(problems))

import torch
# no gpu = no training, stop here
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → T4 GPU."
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "capability", cap, "| torch CUDA", torch.version.cuda)
# compute capability is basically the gpu's "generation number". unsloth needs 7.0 or newer
# T4 = 7.5 (fine), P100 = 6.0 (too old)
assert cap >= (7, 0), "This GPU (e.g. P100) is too old for Unsloth. Switch the accelerator to T4."

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth        2026.11.6
unsloth_zoo    2026.11.6
torch          2.11.0+cu130
transformers   5.17.0
trl            1.15.0
peft           0.21.2
accelerate     1.15.0
bitsandbytes   0.50.2
datasets       4.8.5
GPU: Tesla T4 capability (7, 5) | torch CUDA 13.0


## 3. Load data

In [5]:
import json
import os
import shutil
import glob

# step 1: find where the data files are
if DATA_DIR:
    data_dir = DATA_DIR
elif ON_KAGGLE:
    # kaggle puts datasets somewhere inside /kaggle/input but the exact path depends on the dataset name,
    # so just search for the train file everywhere in there
    hits = sorted(glob.glob(f"/kaggle/input/**/{TRAIN_FILE}", recursive=True))
    assert len(hits) > 0, f"{TRAIN_FILE} not found under /kaggle/input. Add the dataset with 'Add Input'."
    if len(hits) > 1:
        print("several copies found, using the first:", hits)
    data_dir = os.path.dirname(hits[0])
elif COLAB_USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = COLAB_DRIVE_DIR
else:
    # no drive, so upload the 2 files by hand
    from google.colab import files
    data_dir = "/content/data"
    os.makedirs(data_dir, exist_ok=True)
    print(f"Upload {TRAIN_FILE} and {DEV_FILE}")
    uploaded = files.upload()
    for name in uploaded:
        with open(os.path.join(data_dir, name), "wb") as f:
            f.write(uploaded[name])
print("data dir:", data_dir)

# step 2: read them. jsonl = one json object per line
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():   # skip empty lines
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(os.path.join(data_dir, TRAIN_FILE))
dev_rows = read_jsonl(os.path.join(data_dir, DEV_FILE))
print(f"train: {len(train_rows)}  dev: {len(dev_rows)}")

# step 3: look at one example to make sure the format is what we expect
# each row = {"id": ..., "prompt": [system msg, user msg], "completion": [assistant msg]}
ex = train_rows[0]
roles = []
for m in ex["prompt"]:
    roles.append(m["role"])
assert roles == ["system", "user"], "prompt should be system + user"
assert ex["completion"][0]["role"] == "assistant", "completion should be the assistant's answer"
print(ex["prompt"][1]["content"][-400:])   # only the last 400 chars, the schema part is long
print("--- target ---")
print(ex["completion"][0]["content"])

data dir: /kaggle/input/datasets/kshv0532/codegen-nl2mql-sft-approved-v2-1
train: 3220  dev: 225
le [REAL]
- department_management__head
    head_ID: int [INT]
    name: string [TEXT]
    born_state: string [TEXT]
    age: double [REAL]
- department_management__management
    department_ID: int [INT]
    head_ID: int [INT]
    temporary_acting: string [TEXT]

Foreign keys:
(none)

Sample documents:
(none)

Question: How many heads of the departments are older than 56?

Return the JSON object.
--- target ---
{"collection":"department_management__head","pipeline":[{"$match":{"age":{"$gt":56}}},{"$facet":{"rows":[{"$group":{"_id":null,"a0":{"$sum":1}}}]}},{"$project":{"row":{"$ifNull":[{"$arrayElemAt":["$rows",0]},{"a0":0}]}}},{"$replaceRoot":{"newRoot":"$row"}},{"$project":{"_id":0,"count":"$a0"}}]}


## 4. Load the base model (4-bit)

In [6]:
# IMPORTANT: import unsloth BEFORE transformers/trl, it patches them to make them faster
from unsloth import FastLanguageModel
import torch
import time

# load the model + tokenizer in 4 bit (the jpeg thing from config)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # None = auto pick. T4 doesnt support bf16 so it ends up as fp16
    load_in_4bit=True,
)
# how much gpu memory the model is taking (T4 has ~15GB total)
print(torch.cuda.get_device_name(0), f"{torch.cuda.max_memory_reserved()/2**30:.1f} GB reserved after load")

==((====))==  Unsloth 2026.11.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Tesla T4 1.9 GB reserved after load


## 5. Helpers: generation, output check, dev loss

In [7]:
# dict so i can grab any example by its id
by_id = {}
for r in train_rows + dev_rows:
    by_id[r["id"]] = r

dev_ids = set()
for r in dev_rows:
    dev_ids.add(r["id"])

# demo questions HAVE to come from dev (unseen databases), otherwise it's basically cheating
for i in DEMO_IDS:
    assert i in dev_ids, f"{i} is not in the dev split; demo questions must come from unseen databases"


# turns a list of chat messages into token ids using qwen's chat template
# add_generation_prompt=True adds "<|im_start|>assistant" at the end so the model knows it's its turn to talk
def chat_ids(messages, add_generation_prompt):
    # need return_dict=False here, newer transformers returns a dict by default and that breaks things
    ids = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=False,
                                        add_generation_prompt=add_generation_prompt)
    return list(ids)


# ask the model one question and get its answer back as text
def generate(row):
    input_ids = torch.tensor([chat_ids(row["prompt"], True)], device="cuda")
    # no_grad = not training here so dont store gradients (saves memory)
    # do_sample=False = greedy, always picks the most likely token, so same answer every time
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    # out = prompt + answer, so cut off the prompt part and only decode the new stuff
    new_tokens = out[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# checks if the output LOOKS right (valid json, right keys, collection that actually exists)
# this does NOT check if the query is correct!! only running it on atlas can tell that
def shape_check(text, row):
    t = text.strip()
    # the base model likes wrapping everything in ```json ... ``` so strip that first
    if t.startswith("```"):
        t = t.strip("`")
        if "\n" in t:
            t = t[t.find("\n") + 1:]   # removes the "json" word on the first line
    try:
        obj = json.loads(t)
    except Exception:
        return "not valid JSON"
    if not isinstance(obj, dict) or set(obj) != {"collection", "pipeline"}:
        return "JSON, but not exactly {collection, pipeline}"
    # collections show up in the prompt as "- name\n", so check this one is actually in the schema
    if f"- {obj['collection']}\n" not in row["prompt"][1]["content"]:
        return "unknown collection"
    if not isinstance(obj["pipeline"], list):
        return "pipeline is not a list"
    return "ok (valid shape)"


# runs the 3 demo questions and prints model answer vs correct answer
def show_demo(tag):
    FastLanguageModel.for_inference(model)   # puts unsloth in inference mode (faster generation)
    results = {}
    for i in DEMO_IDS:
        row = by_id[i]
        # pull just the question line out of the big prompt (only for printing)
        user_msg = row["prompt"][1]["content"]
        q = user_msg.split("Question:")[-1].strip().splitlines()[0]
        start = time.time()
        out = generate(row)
        dt = time.time() - start
        results[i] = out
        gold = row["completion"][0]["content"]
        print(f"=== [{tag}] {i}  ({dt:.1f}s)\nQ: {q}\nmodel : {out}\ngold  : {gold}\ncheck : {shape_check(out, row)}\n")
    return results


# average loss on the first n dev examples, but ONLY counting the answer tokens
# loss = basically how "surprised" the model is by the correct answer. lower = better
# we only grade the answer part, not the schema/question part
# (like only checking what the student wrote, not the question paper they were given)
def dev_loss(n):
    if n <= 0:
        return None
    FastLanguageModel.for_inference(model)
    losses = []
    for row in dev_rows[:n]:
        prompt_ids = chat_ids(row["prompt"], True)                      # just the question part
        full_ids = chat_ids(row["prompt"] + row["completion"], False)   # question + answer
        # sanity check: the full thing should start with exactly the prompt
        assert full_ids[:len(prompt_ids)] == prompt_ids, "chat template prefix mismatch"
        ids = torch.tensor([full_ids], device="cuda")
        labels = ids.clone()
        # -100 means "ignore this token" in pytorch's loss, so the prompt part doesnt count
        labels[0, :len(prompt_ids)] = -100
        with torch.no_grad():
            loss = model(input_ids=ids, labels=labels).loss
        losses.append(loss.item())
    return sum(losses) / len(losses)

## 6. Before fine-tuning

In [8]:
# ===== BEFORE training =====
# run the untouched base model on the demo questions so we have something to compare against later
before = show_demo("before")
loss_before = dev_loss(DEV_LOSS_SAMPLES)
print("dev loss before:", loss_before)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [before] spider_train_00216  (6.4s)
Q: What are the titles of the books whose writer is not "Elaine Lee"?
model : ```json
{
  "collection": "book_2__book",
  "pipeline": [
    {
      "$match": {
        "Writer": { "$ne": "Elaine Lee" }
      }
    },
    {
      "$project": {
        "_id": 0,
        "Title": 1
      }
    }
  ]
}
```
gold  : {"collection":"book_2__book","pipeline":[{"$match":{"Writer":{"$nin":["Elaine Lee",null]}}},{"$project":{"_id":0,"Title":"$Title"}}]}
check : ok (valid shape)



Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [before] spider_train_00226  (1.4s)
Q: Show different publishers together with the number of publications they have.
model : ```json
{
  "collection": "book_2__publication",
  "pipeline": [
    {
      "$group": {
        "_id": "$Publisher",
        "count": { "$sum": 1 }
      }
    },
    {
      "$sort": { "count": -1 }
    }
  ]
}
```
gold  : {"collection":"book_2__publication","pipeline":[{"$group":{"_id":{"g0":{"$ifNull":["$Publisher",null]}},"a0":{"$sum":1}}},{"$project":{"_id":0,"Publisher":"$_id.g0","count":"$a0"}}]}
check : ok (valid shape)

=== [before] spider_train_00223  (1.4s)
Q: Show writers who have published a book with price more than 4000000.
model : ```json
{
  "collection": "book_2__publication",
  "pipeline": [
    {
      "$match": {
        "Price": { "$gt": 4000000 }
      }
    },
    {
      "$group": {
        "_id": "$Writer"
      }
    }
  ]
}
```
gold  : {"collection":"book_2__book","pipeline":[{"$match":{"Book_ID":{"$ne":null}}},{"$lookup":{"from":

## 7. Format data with the chat template and check lengths

In [9]:
from datasets import Dataset

# turns one example into the full chat text (system + user + assistant) with qwen's special tokens
# like <|im_start|>user ... <|im_end|>  (you can see it in the print below)
def to_text(row):
    messages = row["prompt"] + row["completion"]
    return tokenizer.apply_chat_template(messages, tokenize=False)

def build(rows, name):
    texts = []
    n_tok = []
    for r in rows:
        t = to_text(r)
        texts.append(t)
        n_tok.append(len(tokenizer(t, add_special_tokens=False)["input_ids"]))

    # some length stats so we know MAX_SEQ_LENGTH is big enough
    lens = sorted(n_tok)
    mean_len = sum(lens) / len(lens)
    p99 = lens[int(0.99 * (len(lens) - 1))]   # 99% of examples are shorter than this
    max_len = lens[-1]

    # throw away anything too long (it would get cut off in the middle otherwise)
    keep = []
    for i in range(len(texts)):
        if n_tok[i] <= MAX_SEQ_LENGTH:
            keep.append(texts[i])
    dropped = len(texts) - len(keep)

    print(f"{name}: {len(texts)} rows | tokens mean {mean_len:.0f}, p99 {p99}, "
          f"max {max_len} | dropped (> {MAX_SEQ_LENGTH}): {dropped}")
    return Dataset.from_dict({"text": keep})

train_ds = build(train_rows, "train")
# print the end of one example to see what the model actually trains on
print(train_ds[0]["text"][-500:])

train: 3220 rows | tokens mean 693, p99 1359, max 1447 | dropped (> 2048): 0
ry_acting: string [TEXT]

Foreign keys:
(none)

Sample documents:
(none)

Question: How many heads of the departments are older than 56?

Return the JSON object.<|im_end|>
<|im_start|>assistant
{"collection":"department_management__head","pipeline":[{"$match":{"age":{"$gt":56}}},{"$facet":{"rows":[{"$group":{"_id":null,"a0":{"$sum":1}}}]}},{"$project":{"row":{"$ifNull":[{"$arrayElemAt":["$rows",0]},{"a0":0}]}}},{"$replaceRoot":{"newRoot":"$row"}},{"$project":{"_id":0,"count":"$a0"}}]}<|im_end|>



## 8. Attach LoRA

In [10]:
# stick the LoRA "sticky notes" onto the model (all the settings come from config)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",                            # dont train the bias terms, only the LoRA matrices
    use_gradient_checkpointing="unsloth",   # saves a LOT of memory: forgets some in-between results and recomputes them when needed
    random_state=SEED,
)
# should say ~0.96% trainable. if it says 100% something went wrong
model.print_trainable_parameters()

Unsloth 2026.11.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 9. Train (loss on the assistant's answer only)

In [11]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# all training settings in one dict
cfg = {
    "output_dir": OUTPUT_DIR,
    "dataset_text_field": "text",     # column name in train_ds
    "packing": False,                 # dont glue multiple short examples into one row
    "num_train_epochs": NUM_EPOCHS,
    "max_steps": MAX_STEPS,
    "per_device_train_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_type": LR_SCHEDULER,
    "optim": "adamw_8bit",            # normal adamw but its internal state is stored in 8 bit to save memory
    "weight_decay": 0.01,             # small penalty on big weights, helps a little against overfitting
    "fp16": not is_bfloat16_supported(),   # T4 -> fp16
    "bf16": is_bfloat16_supported(),       # newer gpus (A100 etc) -> bf16
    "logging_steps": 10,              # print the loss every 10 steps
    "save_strategy": "steps",
    "save_steps": 100,                # save a checkpoint every 100 steps in case kaggle dies midway
    "save_total_limit": 2,            # only keep the last 2 checkpoints, they take space
    "seed": SEED,
    "report_to": "none",              # no wandb
    "dataset_num_proc": 2,
}

# TRL keeps renaming arguments between versions which is super annoying,
# so check what this version actually accepts and use that
sft_params = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sft_params:
    cfg["max_length"] = MAX_SEQ_LENGTH
else:
    cfg["max_seq_length"] = MAX_SEQ_LENGTH   # older name

if "warmup_ratio" in sft_params:
    cfg["warmup_ratio"] = WARMUP_RATIO
else:
    # transformers 5 removed warmup_ratio. warmup_steps now takes a float between 0 and 1 as a ratio
    cfg["warmup_steps"] = WARMUP_RATIO

# remove anything this version doesnt know about (otherwise it crashes)
unknown = []
for k in cfg:
    if k not in sft_params:
        unknown.append(k)
if unknown:
    print("not supported by this TRL version, dropped:", unknown)
    for k in unknown:
        del cfg[k]

FastLanguageModel.for_training(model)   # back to training mode (we switched to inference mode for the demo)

# same renaming problem: "tokenizer" became "processing_class" at some point
trainer_args = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_args:
    trainer = SFTTrainer(model=model, train_dataset=train_ds, args=SFTConfig(**cfg), processing_class=tokenizer)
else:
    trainer = SFTTrainer(model=model, train_dataset=train_ds, args=SFTConfig(**cfg), tokenizer=tokenizer)

# only train on the assistant's answer, NOT on the system prompt / schema / question
# we want it to learn to WRITE queries, not memorise schemas
# these are qwen's chat markers, everything before the assistant turn gets masked out of the loss
trainer = train_on_responses_only(trainer,
                                  instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# sanity check: decode only the tokens that actually get trained on. should be just the json answer
lab = trainer.train_dataset[0]["labels"]
trained_tokens = []
for t in lab:
    if t != -100:
        trained_tokens.append(t)
print("trained-on text:", tokenizer.decode(trained_tokens)[:300])

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3220 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/3220 [00:00<?, ? examples/s]

trained-on text: {"collection":"department_management__head","pipeline":[{"$match":{"age":{"$gt":56}}},{"$facet":{"rows":[{"$group":{"_id":null,"a0":{"$sum":1}}}]}},{"$project":{"row":{"$ifNull":[{"$arrayElemAt":["$rows",0]},{"a0":0}]}}},{"$replaceRoot":{"newRoot":"$row"}},{"$project":{"_id":0,"count":"$a0"}}]}<|im_


In [12]:
# ===== TRAIN =====
# this is the long part (~45 min on a T4 for 1 epoch), go do something else
start = time.time()
stats = trainer.train()
train_minutes = (time.time() - start) / 60
print(f"training took {train_minutes:.1f} min | final train loss {stats.training_loss:.4f} | "
      f"peak GPU memory {torch.cuda.max_memory_reserved()/2**30:.1f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,220 | Num Epochs = 1 | Total steps = 403
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.653831
20,0.158614
30,0.080574
40,0.058813
50,0.044826
60,0.039344
70,0.050391
80,0.029732
90,0.022264
100,0.031652


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen25coder3b_lora_r16/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen25coder3b_lora_r16/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen25coder3b_lora_r16/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen25coder3b_lora_r16/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen25coder3b_lora_r16/checkpoint-403/tokenizer_config.json.


training took 43.7 min | final train loss 0.0456 | peak GPU memory 2.7 GB


## 10. After fine-tuning — same questions

In [13]:
# ===== AFTER training =====
# same 3 questions + same dev loss, compare with the "before" numbers
after = show_demo("after")
loss_after = dev_loss(DEV_LOSS_SAMPLES)
print(f"dev loss  before: {loss_before}  after: {loss_after}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [after] spider_train_00216  (1.4s)
Q: What are the titles of the books whose writer is not "Elaine Lee"?
model : {"collection":"book_2__book","pipeline":[{"$match":{"Writer":{"$nin":["Elaine Lee",null]}}},{"$project":{"_id":0,"Title":"$Title"}}]}
gold  : {"collection":"book_2__book","pipeline":[{"$match":{"Writer":{"$nin":["Elaine Lee",null]}}},{"$project":{"_id":0,"Title":"$Title"}}]}
check : ok (valid shape)



Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [after] spider_train_00226  (1.6s)
Q: Show different publishers together with the number of publications they have.
model : {"collection":"book_2__publication","pipeline":[{"$group":{"_id":{"g0":{"$ifNull":["$Publisher",null]}},"a0":{"$sum":1}}},{"$project":{"_id":0,"publisher":"$_id.g0","count":"$a0"}}]}
gold  : {"collection":"book_2__publication","pipeline":[{"$group":{"_id":{"g0":{"$ifNull":["$Publisher",null]}},"a0":{"$sum":1}}},{"$project":{"_id":0,"Publisher":"$_id.g0","count":"$a0"}}]}
check : ok (valid shape)

=== [after] spider_train_00223  (1.2s)
Q: Show writers who have published a book with price more than 4000000.
model : {"collection":"book_2__publication","pipeline":[{"$match":{"Price":{"$gt":4000000}}},{"$project":{"_id":0,"Writer":"$Book_ID"}}]}
gold  : {"collection":"book_2__book","pipeline":[{"$match":{"Book_ID":{"$ne":null}}},{"$lookup":{"from":"book_2__publication","localField":"Book_ID","foreignField":"Book_ID","as":"__j1"}},{"$unwind":"$__j1"},{"$match":{"__j

## 11. Save the adapter
Saves only the LoRA weights (~100 MB for 3B, r=16), plus the settings used. On Kaggle they are in `/kaggle/working/<RUN_NAME>` and become the version's output after a commit run. Load later with `FastLanguageModel.from_pretrained(<adapter dir>, load_in_4bit=True)`.

In [15]:
# save ONLY the LoRA weights (~120MB), not the whole 3B model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import subprocess
import datetime

# checking colab again properly here (kaggle is never colab)
ON_COLAB = (not ON_KAGGLE) and os.path.exists("/var/colab/hostname")
if ON_KAGGLE:
    platform = "kaggle"
elif ON_COLAB:
    platform = "colab"
else:
    platform = "other"

# write down every setting + result so we know what this run was later
# (this is NOT a benchmark, real accuracy only comes from running the queries on atlas)
run_info = {
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": TARGET_MODULES},
    "train": {"epochs": NUM_EPOCHS, "max_steps": MAX_STEPS, "batch": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
              "lr": LEARNING_RATE, "warmup_ratio": WARMUP_RATIO, "scheduler": LR_SCHEDULER, "seed": SEED},
    "data": {"train_rows_used": len(train_ds), "dev_rows": len(dev_rows), "files": [TRAIN_FILE, DEV_FILE]},
    "results_not_a_benchmark": {"train_loss": stats.training_loss, "train_minutes": round(train_minutes, 1),
                                "dev_loss_before": loss_before, "dev_loss_after": loss_after,
                                "dev_loss_samples": DEV_LOSS_SAMPLES},
    "demo": {},
    "gpu": torch.cuda.get_device_name(0),
    "platform": platform,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
for i in DEMO_IDS:
    run_info["demo"][i] = {"before": before[i], "after": after[i]}

with open(os.path.join(OUTPUT_DIR, "run_info.json"), "w") as f:
    json.dump(run_info, f, indent=1)

# save exact package versions so teammates can copy the same setup
with open(os.path.join(OUTPUT_DIR, "pip_freeze.txt"), "w") as f:
    freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True)
    f.write(freeze.stdout)

# colab wipes everything when the session ends, so copy to drive (skipping checkpoints, too big)
if ON_COLAB and COLAB_USE_DRIVE and COLAB_COPY_TO_DRIVE and data_dir.startswith("/content/drive"):
    dest = os.path.join(os.path.dirname(data_dir.rstrip("/")), "runs", RUN_NAME)
    shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
    print("copied to", dest)
print("saved to", OUTPUT_DIR)
!ls -la {OUTPUT_DIR}

saved to /kaggle/working/qwen25coder3b_lora_r16
total 128212
drwxr-xr-x 4 root root      4096 Nov 19 04:46 .
drwxr-xr-x 6 root root      4096 Nov 19 03:56 ..
-rw-r--r-- 1 root root      1334 Nov 19 04:46 adapter_config.json
-rw------- 1 root root 119801528 Nov 19 04:46 adapter_model.safetensors
-rw-r--r-- 1 root root      2507 Nov 19 04:46 chat_template.jinja
drwxr-xr-x 2 root root      4096 Nov 19 04:40 checkpoint-400
drwxr-xr-x 2 root root      4096 Nov 19 04:40 checkpoint-403
-rw-r--r-- 1 root root     18289 Nov 19 04:46 pip_freeze.txt
-rw-r--r-- 1 root root      1585 Nov 19 04:46 README.md
-rw-r--r-- 1 root root      2389 Nov 19 04:46 run_info.json
-rw-r--r-- 1 root root      4907 Nov 19 04:46 tokenizer_config.json
-rw-r--r-- 1 root root  11422356 Nov 19 04:46 tokenizer.json


## Things to try later
- `LORA_R` 8 / 32, `LEARNING_RATE` 1e-4, `NUM_EPOCHS` 2
- the 7B model (expect roughly 2–3× the training time; drop `BATCH_SIZE` to 1 and raise `GRAD_ACCUM` to 8 if you run out of memory)
- different `DEMO_IDS` (any id from the dev file)

Lower dev loss and valid-shape outputs mean the run worked. They do **not** mean the queries return the right
rows; only Atlas execution with the project scorer can say that.

Also training time took ~40 mins for one SINGLE EPOCH, see if you can utilize both T4 GPUs for massive speedup?